In [1]:
import galsim
import jax.numpy as jnp
import jax_galsim as xgalsim
from jax import random
from jax._src.prng import PRNGKeyArray
from jax.typing import ArrayLike
from jax_galsim import GSParams

from functools import partial

import jax 

In [2]:
def draw_gaussian_gaussian_psf(
    *,
    f: float,
    hlr: float,
    e1: float,
    e2: float,
    x: float,  # pixels
    y: float,
    slen: int,
    fft_size: int,  # rule of thumb: at least 4 times `slen`
    psf_fwhm: float = 0.8,
    pixel_scale: float = 0.2,
):
    gsparams = GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)

    gal = xgalsim.Gaussian(flux=f, half_light_radius=hlr)
    gal = gal.shear(g1=e1, g2=e2)

    psf = xgalsim.Gaussian(flux=1.0, fwhm=0.8)
    gal_conv = xgalsim.Convolve([gal, psf]).withGSParams(gsparams)
    image = gal_conv.drawImage(nx=slen, ny=slen, scale=pixel_scale, offset=(x, y))
    return image.array

In [3]:
def draw_gaussian_moffat_psf(
    *,
    f: float,
    hlr: float,
    e1: float,
    e2: float,
    x: float,  # pixels
    y: float,
    slen: int,
    fft_size: int,  # rule of thumb: at least 4 times `slen`
    psf_fwhm: float = 0.8,
    pixel_scale: float = 0.2,
):
    gsparams = GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)

    gal = xgalsim.Gaussian(flux=f, half_light_radius=hlr)
    gal = gal.shear(g1=e1, g2=e2)

    psf = xgalsim.Moffat(flux=1.0, scale_radius=0.8, beta=2.0)
    gal_conv = xgalsim.Convolve([gal, psf]).withGSParams(gsparams)
    image = gal_conv.drawImage(nx=slen, ny=slen, scale=pixel_scale, offset=(x, y))
    return image.array

In [12]:
_func1 = jax.jit(partial(draw_gaussian_gaussian_psf, slen=73, fft_size=256))
_ = _func1(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

In [13]:
%%timeit
_func1(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

233 μs ± 7.61 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
_func2 = jax.jit(partial(draw_gaussian_moffat_psf, slen=63, fft_size=256))
_ = _func2(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

In [15]:
%%timeit
_func2(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0) 

29.4 ms ± 75.3 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
